# 5G NR transport block

> CRC, LDPC and rate matching of 5G NR with Sionna PHY, and the decoding back.

In [ ]:
#| default_exp coding.nr

In [ ]:
#| hide
from nbdev.showdoc import *

The channel code of 5G NR's shared channels (PUSCH, PDSCH) with [Sionna PHY](https://nvlabs.github.io/sionna/phy/index.html):
`NRTBEncoder` adds the transport block's CRC, splits it into code blocks with their own CRCs, LDPC-encodes them
and rate-matches them to the coded bits available; `NRTBDecoder` decodes the LLRs and checks the CRC (`crc_ok`).
They wrap Sionna's `TBEncoder` and `TBDecoder`, built once for each transport block.

What they code is set by the group's parameters, `tb_size`, `num_coded_bits`, `coderate` and
`num_bits_per_symbol`: link adaptation chooses them (`NRLinkAdaptation`, see `radio.link_adaptation`), so coding
knows nothing about MCS tables. Code rates below 1/5 (`MIN_LDPC_CODERATE`) are not supported by Sionna's encoder.

It needs Sionna (PyTorch): `pip install 'comm-core[sionna]'`.

In [ ]:
#| export
from __future__ import annotations

import numpy as np

from comm_core.block import Block

In [ ]:
#| export
MIN_LDPC_CODERATE = 0.2                                  # Sionna's 5G LDPC encoder: rates below 1/5 are not supported


class NRTBEncoder(Block):
    "5G NR transport block coding with Sionna PHY (`TBEncoder`): CRC, LDPC and rate matching, for each group's transport block."

    outputs = ('codeword',)

    def __init__(
        self,
        channel_type: str = 'PUSCH', # 'PUSCH' (uplink) or 'PDSCH' (downlink)
        num_bp_iter: int = 20, # LDPC decoder iterations (of the decoder built with each encoder)
        device: str | None = None, # Torch device; None: Sionna's default (a GPU if there is one)
        precision: str = 'single', # 'single' or 'double'
    ):
        import torch
        self._torch, self.channel_type, self.num_bp_iter = torch, channel_type, num_bp_iter
        self.device, self.precision = device, precision
        self.real = torch.float64 if precision == 'double' else torch.float32
        self._coders = {}                                  # (TB size, coded bits, rate, bits per symbol) -> (encoder, decoder)

    def coders(
        self,
        tb_size: int, # Transport block size
        num_coded_bits: int, # Coded bits available (resource elements × bits per symbol)
        coderate: float, # Target code rate
        num_bits_per_symbol: int, # Bits per QAM symbol
    ) -> tuple:
        "(encoder, decoder) of a transport block: Sionna's `TBEncoder` and `TBDecoder`, built once."
        key = (int(tb_size), int(num_coded_bits), float(coderate), int(num_bits_per_symbol))
        if key not in self._coders:
            import contextlib, io
            from sionna.phy.nr import TBDecoder, TBEncoder
            with contextlib.redirect_stdout(io.StringIO()):
                encoder = TBEncoder(target_tb_size=key[0], num_coded_bits=key[1], target_coderate=key[2],
                                    num_bits_per_symbol=key[3], channel_type=self.channel_type,
                                    precision=self.precision, device=self.device)
            decoder = TBDecoder(encoder, num_bp_iter=self.num_bp_iter, precision=self.precision, device=self.device)
            self._coders[key] = (encoder, decoder)
        return self._coders[key]

    def call(
        self,
        bits, # The payload bits, (B, ..., k) with k <= tb_size (zero-padded to the transport block)
        tb_size: int, # Transport block size
        num_coded_bits: int, # Coded bits available
        coderate: float, # Target code rate
        num_bits_per_symbol: int, # Bits per QAM symbol
    ):
        "The codewords, (B, num_coded_bits), a torch tensor."
        encoder, _ = self.coders(tb_size, num_coded_bits, coderate, num_bits_per_symbol)
        flat = np.asarray(bits).reshape(len(bits), -1)
        padded = np.zeros((len(bits), encoder.k))          # the payload, zero-padded to the transport block
        padded[:, :flat.shape[1]] = flat
        return encoder(self._torch.as_tensor(padded, dtype=self.real, device=encoder.device))


class NRTBDecoder(Block):
    "5G NR transport block decoding with Sionna PHY (`TBDecoder`): LDPC decoding of the LLRs, then the CRC."

    outputs = ('bits_hat', 'crc_ok')

    def __init__(
        self,
        encoder: NRTBEncoder, # Its encoder: the decoder of each transport block is built with it
    ):
        self.encoder = encoder

    def children(self) -> list:
        "None: the encoder is a stage of its own."
        return []

    def call(
        self,
        llr, # LLRs of the codewords, (B, num_coded_bits)
        tb_size: int, # As the encoder's
        num_coded_bits: int,
        coderate: float,
        num_bits_per_symbol: int,
    ) -> tuple:
        "(bits_hat, crc_ok): the transport block's bits, (B, tb_size) numpy, and whether its CRC holds, (B,)."
        _, decoder = self.encoder.coders(tb_size, num_coded_bits, coderate, num_bits_per_symbol)
        bits, crc = decoder(llr)
        return bits.cpu().numpy().round().astype(np.int64), crc.cpu().numpy()

In [ ]:
show_doc(NRTBEncoder.coders)

In [ ]:
import torch

from comm_core.radio.link_adaptation import nr_allocation

In [ ]:
#| sionna
# a 1000-bit payload at 16-QAM, rate 1/2: decoded exactly from clean LLRs, and lost (CRC) from random ones
encoder = NRTBEncoder(device='cpu')
decoder = NRTBDecoder(encoder)
rng = np.random.default_rng(0)
bits = rng.integers(0, 2, (3, 1000))
num_re, tb_size = nr_allocation(1000, 4, 0.5)
params = dict(tb_size=tb_size, num_coded_bits=num_re * 4, coderate=0.5, num_bits_per_symbol=4)
cw = encoder(bits, **params)
assert tuple(cw.shape) == (3, num_re * 4)
bits_hat, ok = decoder(20.0 * (2 * cw - 1), **params)                    # LLR > 0 for a 1
assert ok.all() and (bits_hat[:, :1000] == bits).all()
_, ok = decoder(torch.as_tensor(rng.standard_normal((3, num_re * 4)), dtype=torch.float32), **params)
assert not ok.any()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()